# 🎬 OpenShorts AI Studio no Google Colab (Aceleração por GPU / CUDA)

Este notebook executa a interface visual completa do **OpenShorts AI Studio** (com Gradio WebUI + FastAPI Backend) utilizando a GPU do Google Colab (T4 / V100 / A100).

### 🌟 Recursos Integrados:
- 🎨 **Interface Visual Gradio:** Geração de cortes 9:16 com preview de vídeo e download.
- 📺 **Auto-Channel Watcher:** Monitore canais do YouTube e corte automaticamente.
- 📅 **Smart Scheduler:** Enfileire posts nos horários de maior engajamento.
- ⚡ **Suporte à Extensão Chrome:** Gera URL pública para disparar cortes com 1 clique.

---

### 1. Verificar GPU Ativa (CUDA)

In [ ]:
!nvidia-smi

### 2. Autenticação Interativa & Clone do Repositório Privado
Execute a célula abaixo para autenticar com seu GitHub Token e Gemini API Key:

In [ ]:
import os
import getpass
from google.colab import userdata

# 1. Obter GitHub Personal Access Token (PAT)
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None

if not github_token:
    print("🔑 Autenticação GitHub:")
    github_token = getpass.getpass("Digite ou cole seu GitHub Personal Access Token (PAT): ").strip()

# 2. Obter Gemini API Key
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_key = None

if not gemini_key:
    print("\n🤖 Autenticação Gemini AI:")
    gemini_key = getpass.getpass("Digite ou cole sua Gemini API Key (Google AI Studio): ").strip()

# 3. Opcional: Cookies do YouTube para evitar bloqueio bot do Google no Colab
try:
    yt_cookies = userdata.get('YOUTUBE_COOKIES')
except Exception:
    yt_cookies = None

if yt_cookies:
    os.environ['YOUTUBE_COOKIES'] = yt_cookies

# Validar chaves
if not github_token:
    raise ValueError("❌ O GitHub Token é obrigatório para acessar o repositório privado.")
if not gemini_key:
    raise ValueError("❌ A Gemini API Key é obrigatória para processar os vídeos.")

# Configurar variáveis de ambiente do sistema
os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['WHISPER_DEVICE'] = 'cuda'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16'
os.environ['MAX_CONCURRENT_JOBS'] = '2'

# Clonar ou atualizar o repositório privado
repo_url = f"https://{github_token}@github.com/diegofullstackjs/new-open-shorts.git"
target_dir = "/content/new-open-shorts"

if not os.path.exists(target_dir):
    print("\n🚀 Clonando repositório privado new-open-shorts...")
    !git clone {repo_url} {target_dir}
else:
    print("\n🔄 Atualizando repositório existente...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
print("\n✅ Autenticação realizada e repositório carregado com sucesso!")

### 3. Instalar Dependências do Sistema e Pacotes Python

In [ ]:
# Instalar pacotes de sistema necessários
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-noto fonts-noto-cjk libgl1-mesa-glx

# Atualizar yt-dlp para a versão mais recente e instalar dependências
!pip install -U --no-cache-dir yt-dlp
!pip install -q -r requirements.txt
!pip install -q pycloudflared pyngrok uvicorn

### 4. Iniciar a Interface Visual Gradio WebUI (com Link Público Compartilhável)
Inicia a interface gráfica completa do OpenShorts e gera um link público `.gradio.live` para você usar no navegador ou no celular!

In [ ]:
# Iniciar o Gradio App com túnel público automático (share=True)
!python gradio_app.py

### 5. (Opcional) Iniciar Servidor FastAPI + Cloudflare Tunnel para a Extensão Chrome
Se você for usar a **Extensão Chrome 1-Click Shortify**, execute a célula abaixo para obter a URL da API:

In [ ]:
from pycloudflared import try_cloudflare

tunnel_url = try_cloudflare(port=8000)
print('='*75)
print(f'🚀 URL PÚBLICA DA API DO OPENSHORTS: {tunnel_url.tunnel}')
print('👉 Cole esta URL nas configurações da sua Extensão Chrome 1-Click Shortify!')
print('='*75)

!python -m uvicorn app:app --host 0.0.0.0 --port 8000